# 第 15 课 · 评估改进是否可信

博士训练中的统计判断要延伸到模型评估：同一测试集上的模型差异是配对数据；同一模型多次采样不是多个独立训练种子；反复调参会消耗测试集的独立性。

本课用第 14 课实际保存的逐题预测计算配对 bootstrap 区间，并用第 12 课“0/20 正确”说明零观察值的含义。需要先运行第 14 课的 SFT 评估。使用 CPU 即可。

## 1 · 先检查数据单位与评估协议

这里的单位是数值相同的一道题。同题的前导零、交换顺序、多个采样回答不能直接当作独立题目。复杂数据应按用户、文档或题目族分组重采样。

本实验 test 是固定的小型三位数集合。下面的区间仅反映在近似独立、代表性抽样假设下的题目抽样不确定性；不包含训练种子、设备差异或真实业务分布变化。


In [ ]:
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
path = Path('support/experiment14_output/predictions.csv')
assert path.exists(), '请先完成第 14 课第 3 节'
df = pd.read_csv(path, keep_default_na=False)
df = df[df.split=='test'].copy()
assert not df.duplicated(['stage','number']).any()
def bool_values(series):
    result = series.astype(str).str.lower().map({'true':1, 'false':0, '1':1, '0':0})
    assert result.notna().all(), '未知布尔值'
    return result.astype(int)
df['score'] = bool_values(df.correct)
paired = df.pivot(index='number', columns='stage', values='score')
assert {'cold','sft'} <= set(paired.columns) and not paired[['cold','sft']].isna().any().any()
display(paired.head())
print('测试题数:', len(paired))


## 2 · 配对差异与 bootstrap

先对每题计算 $d_i=score_{SFT,i}-score_{cold,i}$，再重采样题目索引。两个模型必须使用同一组重采样索引；分别采样会丢失题目难度带来的相关性。

本格使用 10,000 次重采样和固定种子，报告百分位区间。小样本或所有差值都相同时，bootstrap 可能退化为很窄甚至零宽区间，这不意味着未来样本没有不确定性。不要用一个区间替代实验设计。


In [ ]:
delta = (paired.sft-paired.cold).to_numpy(dtype=float)
rng = np.random.default_rng(15015)
indices = rng.integers(0, len(delta), size=(10000,len(delta)))
boot = delta[indices].mean(axis=1)
lo, hi = np.quantile(boot, [.025,.975])
improved = int((delta>0).sum()); worsened = int((delta<0).sum())
print(f'准确率差: {delta.mean():.1%}; 配对 bootstrap 95% 区间: [{lo:.1%}, {hi:.1%}]')
print('改善题数:', improved, '退化题数:', worsened, '不变:', int((delta==0).sum()))
# 明确的边界检查：相同模型在每道题上差值均为零。
same = (paired.sft-paired.sft).to_numpy()
assert same[indices].mean() == 0


## 3 · 0/20 正确并不等于总体成功概率严格为零

假定题目是独立同分布的 Bernoulli 试验，观察 n 次全失败后，成功概率的单侧 95% 精确上界为 $1-0.05^{1/n}$。对第 12 课 n=20 的结果，这个界说明小样本的分辨率有限，而不是替失败结果开脱。

Wilson 区间可用于单个模型的二项比例描述，不能把两个模型的 Wilson 区间是否重叠当成配对差异检验。


In [ ]:
def wilson(successes, total, z=1.959963984540054):
    assert 0 <= successes <= total and total > 0
    p = successes/total
    center = (p+z*z/(2*total))/(1+z*z/total)
    radius = z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/(1+z*z/total)
    return max(0, center-radius), min(1, center+radius)
upper = 1-.05**(1/20)
print(f'0/20 的单侧 95% 精确上界: {upper:.2%}')
for stage in ['cold','sft']:
    successes = int(paired[stage].sum())
    print(stage, successes, '/', len(paired), 'Wilson:', wilson(successes,len(paired)))
assert wilson(0,20)[0] < 1e-12
assert wilson(20,20)[1] > 1-1e-12
out=Path('support/experiment15_output'); out.mkdir(exist_ok=True)
report=dict(n=len(delta), delta=float(delta.mean()), paired_bootstrap_95=[float(lo),float(hi)],
    improved=improved, worsened=worsened, zero_of_20_one_sided_95_upper=upper,
    scope='Item uncertainty only; one training seed; approximate iid item assumption.')
(out/'evaluation_audit.json').write_text(json.dumps(report,indent=2))


## 4 · 从单次演示到可发表、可上线的证据

在运行前写好实验卡：问题、主指标、数据分组、基线、预算、停止条件和失败判据。验证集用于选方案，最终测试只回答预先指定的问题。公开教程里的测试集一旦被反复查看，就应视为开发基准。

| 问题 | 应记录什么 | 常见误判 |
|---|---|---|
| 训练是否稳定 | 每个训练种子的结果、均值与离散程度 | 把多个采样回答当成多个种子 |
| 哪些用户受损 | 按题型/难度/长度分层的指标与样本数 | 只报总体均值 |
| RL 是否值得成本 | 同起点冻结、继续 SFT、RL，GPU 时间与 token 预算 | 只比较更新步数 |
| 线上是否改善 | 预先定义的业务指标、延迟、成本与回滚阈值 | 离线分数直接等于用户收益 |

毕业作业：至少三个训练种子，对同一最终题集保存预测；分别报告种子变异与题目不确定性。若要合并估计，说明层级重采样结构，不把所有 seed×题目行当成独立观测。保留每次运行，而不是挑最好的种子。
